# 04 — Acquire Eurostat oil balance

Pull harmonised annual oil supply/transformation/consumption observations for Portugal and Spain from `nrg_cb_oil`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.eurostat import build_balance_panel, canonicalise_oil_balance, fetch_jsonstat, jsonstat_to_frame


In [ ]:
frames = []
for geo in ["PT", "ES"]:
    payload = fetch_jsonstat("nrg_cb_oil", params={"geo": geo, "lang": "en"})
    frame = jsonstat_to_frame(payload)
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True)
print(raw.shape)
display(raw.head())
persist_dataframe(raw, PATHS.interim / "eurostat_nrg_cb_oil_pt_es.csv")


In [ ]:
canonical = canonicalise_oil_balance(raw)
if canonical.empty:
    raise RuntimeError("Eurostat oil-balance extraction produced no diesel/gasoline balance terms; inspect labels above before proceeding.")
persist_dataframe(
    canonical,
    PATHS.interim / "eurostat_oil_balance_canonical.csv",
    key_columns=["year", "country", "product", "flow"],
    metadata={"unit": "kt"},
)
balance_panel = build_balance_panel(canonical)
persist_dataframe(
    balance_panel,
    PATHS.processed / "eurostat_physical_balance_panel.csv",
    key_columns=["year", "country", "product"],
)
display(balance_panel.head())


In [ ]:
# Inspect labels before hard-coding selections. This makes Eurostat code changes visible.
for column in ["siec_label", "nrg_bal_label", "unit_label"]:
    if column in raw.columns:
        print(f"\n{column}")
        display(pd.Series(sorted(raw[column].dropna().astype(str).unique())).head(80))
